In [ ]:
import sqlite3
from datetime import datetime

timestamp_ns = 1715630000123456789

conn = sqlite3.connect("rosbags/slalom_lite_2/a_slalom_lite_zed_2_0.db3")
cur = conn.cursor()

In [ ]:
cur.execute("SELECT * FROM topics")
for row in cur.fetchall():
    print(row)

In [ ]:

cur.execute("""
            SELECT topics.name, COUNT(messages.id)
            FROM messages
                     JOIN topics ON messages.topic_id = topics.id
            GROUP BY topics.name
            """)

for row in cur.fetchall():
    print(row)

In [ ]:
cur.execute("""
            SELECT timestamp
            FROM messages
            ORDER BY timestamp
                LIMIT 200

            """)


In [ ]:
for row in cur.fetchall():
    dt = datetime.fromtimestamp(row[0] / 1e9)
    print(dt)

In [ ]:
conn.close()

In [ ]:
import cv2
import rosbag2_py
from cv_bridge import CvBridge
from rclpy.serialization import deserialize_message
from rosidl_runtime_py.utilities import get_message

BAG_PATH = "rosbags/slalom_lite_2"  # dossier du bag, pas juste le .db3
IMAGE_TOPIC = "/zed/zed_node/rgb/image_rect_color"  # à changer au besoin
OUTPUT_VIDEO = "slalom_lite_zed_2.mp4"
FPS = 25.0

reader = rosbag2_py.SequentialReader()

storage_options = rosbag2_py.StorageOptions(
    uri=BAG_PATH,
    storage_id="sqlite3"
)

converter_options = rosbag2_py.ConverterOptions(
    input_serialization_format="cdr",
    output_serialization_format="cdr"
)

reader.open(storage_options, converter_options)

topics = reader.get_all_topics_and_types()
type_map = {topic.name: topic.type for topic in topics}

print("Topics disponibles:")
for topic in topics:
    print(f"{topic.name} -> {topic.type}")

if IMAGE_TOPIC not in type_map:
    raise ValueError(f"Topic introuvable: {IMAGE_TOPIC}")

msg_type = get_message(type_map[IMAGE_TOPIC])
bridge = CvBridge()

video_writer = None
frame_count = 0

while reader.has_next():
    topic, data, timestamp = reader.read_next()

    if topic != IMAGE_TOPIC:
        continue

    msg = deserialize_message(data, msg_type)

    if type_map[IMAGE_TOPIC] == "sensor_msgs/msg/CompressedImage":
        frame = bridge.compressed_imgmsg_to_cv2(msg)
    else:
        frame = bridge.imgmsg_to_cv2(msg, desired_encoding="bgr8")

    if video_writer is None:
        height, width = frame.shape[:2]

        video_writer = cv2.VideoWriter(
            OUTPUT_VIDEO,
            cv2.VideoWriter_fourcc(*"mp4v"),
            FPS,
            (width, height)
        )

        print(f"Video: {width}x{height} @ {FPS} FPS")

    video_writer.write(frame)
    frame_count += 1

    if frame_count % 500 == 0:
        print(f"{frame_count} frames écrites...")

if video_writer is not None:
    video_writer.release()

print(f"Terminé: {frame_count} frames écrites dans {OUTPUT_VIDEO}")

In [1]:
import sqlite3

INPUT_DB = "rosbags/slalom_lite_2/a_slalom_lite_zed_2_0.db3"
OUTPUT_DB = "rosbags/slalom_lite_2/slalom_lite_zed_2_0.db3"

IMAGE_TOPIC = "/zed/zed_node/rgb/image_rect_color"

# -----------------------------------
# Connexions
# -----------------------------------

src_conn = sqlite3.connect(INPUT_DB)
src_cur = src_conn.cursor()

dst_conn = sqlite3.connect(OUTPUT_DB)
dst_cur = dst_conn.cursor()

# -----------------------------------
# Recréer les tables ROS2
# -----------------------------------

dst_cur.execute("""
                CREATE TABLE topics
                (
                    id                   INTEGER PRIMARY KEY,
                    name                 TEXT NOT NULL,
                    type                 TEXT NOT NULL,
                    serialization_format TEXT NOT NULL,
                    offered_qos_profiles TEXT NOT NULL
                );
                """)

dst_cur.execute("""
                CREATE TABLE messages
                (
                    id        INTEGER PRIMARY KEY,
                    topic_id  INTEGER NOT NULL,
                    timestamp INTEGER NOT NULL,
                    data      BLOB    NOT NULL
                );
                """)

# -----------------------------------
# Copier les topics
# -----------------------------------

src_cur.execute("SELECT * FROM topics")
topics = src_cur.fetchall()

dst_cur.executemany("""
                    INSERT INTO topics
                    VALUES (?, ?, ?, ?, ?)
                    """, topics)

# -----------------------------------
# Trouver le topic image
# -----------------------------------

src_cur.execute("""
                SELECT id
                FROM topics
                WHERE name = ?
                """, (IMAGE_TOPIC,))

topic_id = src_cur.fetchone()[0]

# -----------------------------------
# Sélectionner environ 1 image/seconde
# -----------------------------------

src_cur.execute("""
                SELECT id, topic_id, timestamp, data
                FROM (
                    SELECT
                    id, topic_id, timestamp, data, timestamp / 1000000000 AS second_bucket, ROW_NUMBER() OVER (
                    PARTITION BY timestamp / 1000000000
                    ORDER BY timestamp
                    ) AS rn
                    FROM messages
                    WHERE topic_id = ?
                    )
                WHERE rn = 1
                ORDER BY timestamp;
                """, (topic_id,))

rows = src_cur.fetchall()

print(f"Frames conservées: {len(rows)}")

# -----------------------------------
# Écrire dans le nouveau bag
# -----------------------------------

dst_cur.executemany("""
                    INSERT INTO messages(id, topic_id, timestamp, data)
                    VALUES (?, ?, ?, ?)
                    """, rows)

dst_conn.commit()

print("Nouveau bag créé:", OUTPUT_DB)

# -----------------------------------
# Fermer
# -----------------------------------

src_conn.close()
dst_conn.close()

Frames conservées: 173
Nouveau bag créé: rosbags/slalom_lite_2/slalom_lite_zed_2_1fps.db3
